In [12]:
# imports
import os
from statsmodels.stats.multitest import multipletests

from src.config import PATH_FIG
from src.analysis.utils import color_min_max, build_latex_table, fmt_sci
from src.analysis.moralchoice_analysis import calculate_moralchoice_results
from src.analysis.normbank_analysis import calculate_normbank_results
from src.analysis.reddit_analysis import calculate_reddit_verdict_results

OUTPUT_DIR = PATH_FIG / "tables"

In [13]:
MODELS = ["gemma-3-4b-it", "Llama-3.2-3B-Instruct", "Qwen3-4B", "GPT-4.1"]
MODEL_LABEL = {
    "gemma-3-4b-it":         "\\lm{gemma-3-4b-it}",
    "Llama-3.2-3B-Instruct": "\\lm{Llama-3.2-3B-Instruct}",
    "Qwen3-4B":              "\\lm{Qwen3-4B}",
    "GPT-4.1":               "\\lm{GPT-4.1}",
}

# (dataset_key, figure_name, call_fn, positional_args)
dataset_configs = {
    "moralchoice_high_ambiguity": [
        ("gemma-3-4b-it",         calculate_moralchoice_results,
         ("main/moralchoice_high_ambiguity/google_gemma-3-4b-it_moralchoice_high_ambiguity.csv",
          "moralchoice_high_ambiguity.csv")),
        ("Llama-3.2-3B-Instruct", calculate_moralchoice_results,
         ("main/moralchoice_high_ambiguity/meta-llama_Llama-3.2-3B-Instruct_moralchoice_high_ambiguity.csv",
          "moralchoice_high_ambiguity.csv")),
        ("Qwen3-4B",              calculate_moralchoice_results,
         ("main/moralchoice_high_ambiguity/Qwen_Qwen3-4B_moralchoice_high_ambiguity.csv",
          "moralchoice_high_ambiguity.csv")),
        ("GPT-4.1",               calculate_moralchoice_results,
         ("main/moralchoice_high_ambiguity/openai_gpt-4.1_moralchoice_high_ambiguity.csv",
          "moralchoice_high_ambiguity.csv")),
        ("gemma-3-4b-it-visual",  calculate_moralchoice_results,
         ("multimodal/google_gemma-3-4b-it_multimodality_high_ambiguity.csv",
         "moralchoice_high_ambiguity.csv")),
        ("Qwen3-VL-4B-Instruct",  calculate_moralchoice_results,
         ("multimodal/Qwen_Qwen3-VL-4B-Instruct_moralchoice_high_ambiguity.csv",
         "moralchoice_high_ambiguity.csv"))
    ],
    "moralchoice_low_ambiguity": [
        ("gemma-3-4b-it",         calculate_moralchoice_results,
         ("main/moralchoice_low_ambiguity/google_gemma-3-4b-it_moralchoice_low_ambiguity.csv",
          "moralchoice_low_ambiguity.csv")),
        ("Llama-3.2-3B-Instruct", calculate_moralchoice_results,
         ("main/moralchoice_low_ambiguity/meta-llama_Llama-3.2-3B-Instruct_moralchoice_low_ambiguity.csv",
          "moralchoice_low_ambiguity.csv")),
        ("Qwen3-4B",              calculate_moralchoice_results,
         ("main/moralchoice_low_ambiguity/Qwen_Qwen3-4B_moralchoice_low_ambiguity.csv",
          "moralchoice_low_ambiguity.csv")),
        ("GPT-4.1",               calculate_moralchoice_results,
         ("main/moralchoice_low_ambiguity/openai_gpt-4.1_moralchoice_low_ambiguity.csv",
          "moralchoice_low_ambiguity.csv")),
        ("gemma-3-4b-it-visual",  calculate_moralchoice_results,
         ("multimodal/google_gemma-3-4b-it_multimodality_low_ambiguity.csv",
         "moralchoice_low_ambiguity.csv")),
        ("Qwen3-VL-4B-Instruct",  calculate_moralchoice_results,
         ("multimodal/Qwen_Qwen3-VL-4B-Instruct_moralchoice_low_ambiguity.csv",
         "moralchoice_low_ambiguity.csv"))
    ],
    "normbank": [
        ("gemma-3-4b-it",         calculate_normbank_results,
         ("main/normbank/google_gemma-3-4b-it_normbank.csv",)),
        ("Llama-3.2-3B-Instruct", calculate_normbank_results,
         ("main/normbank/meta-llama_Llama-3.2-3B-Instruct_normbank.csv",)),
        ("Qwen3-4B",              calculate_normbank_results,
         ("main/normbank/Qwen_Qwen3-4B_normbank.csv",)),
        ("GPT-4.1",               calculate_normbank_results,
         ("main/normbank/openai_gpt-4.1_normbank.csv",)),
    ],
    "reddit": [
        ("gemma-3-4b-it",         calculate_reddit_verdict_results,
         ("main/reddit/google_gemma-3-4b-it_reddit.csv",)),
        ("Llama-3.2-3B-Instruct", calculate_reddit_verdict_results,
         ("main/reddit/meta-llama_Llama-3.2-3B-Instruct_reddit.csv",)),
        ("Qwen3-4B",              calculate_reddit_verdict_results,
         ("main/reddit/Qwen_Qwen3-4B_reddit.csv",)),
        ("GPT-4.1",               calculate_reddit_verdict_results,
         ("main/reddit/openai_gpt-4.1_reddit.csv",)),
    ],
}

all_results = {}

for dataset, entries in dataset_configs.items():
    all_results[dataset] = {}
    for model_key, fn, args in entries:
        print(f"  Running {dataset} / {model_key} ...", end=" ")
        model_results = fn(*args, sig_testing=True)
        print("done")
        for option, option_results in model_results.items():
            if option not in all_results[dataset]:
                all_results[dataset][option] = {}
            all_results[dataset][option][model_key] = option_results
            print(f"    {option}\n")
            print(f"    scores: {option_results["mean_scores"]}")
            print(f"    sig:    {option_results["sig"]}\n")

  Running moralchoice_high_ambiguity / gemma-3-4b-it ... done
    all

    scores: {'baseline': np.float64(0.6382800039555618), 'positive': np.float64(0.6589659445560377), 'neutral': np.float64(0.6529429080751802), 'negative': np.float64(0.6286993957229956)}
    sig:    {'baseline_vs_positive': {'coef': np.float64(0.020685940600476183), 'stat': np.float64(3.801113067052377), 'pvalue': np.float64(0.00014404754749379334), 'ref': 'baseline', 'test': 'positive'}, 'baseline_vs_neutral': {'coef': np.float64(0.014662904119617905), 'stat': np.float64(2.6562307076228096), 'pvalue': np.float64(0.007901953405358412), 'ref': 'baseline', 'test': 'neutral'}, 'baseline_vs_negative': {'coef': np.float64(-0.009580608232565914), 'stat': np.float64(-1.6828320734105076), 'pvalue': np.float64(0.09240760417529886), 'ref': 'baseline', 'test': 'negative'}, 'positive_vs_neutral': {'coef': np.float64(0.006023036480857628), 'stat': np.float64(2.59399513355994), 'pvalue': np.float64(0.009486782970000295), 'ref': 

C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users

done
    all

    scores: {'baseline': np.float64(0.9798399319374549), 'positive': np.float64(0.9905059688822787), 'neutral': np.float64(0.9878710631429924), 'negative': np.float64(0.8609348500121338)}
    sig:    {'baseline_vs_positive': {'coef': np.float64(0.010666036944819185), 'stat': np.float64(8.17174304771046), 'pvalue': np.float64(3.039653964336313e-16), 'ref': 'baseline', 'test': 'positive'}, 'baseline_vs_neutral': {'coef': np.float64(0.00803113120553355), 'stat': np.float64(5.904907361008636), 'pvalue': np.float64(3.528449670076196e-09), 'ref': 'baseline', 'test': 'neutral'}, 'baseline_vs_negative': {'coef': np.float64(-0.118905081925343), 'stat': np.float64(-22.04284781001238), 'pvalue': np.float64(1.118745552741771e-107), 'ref': 'baseline', 'test': 'negative'}, 'positive_vs_neutral': {'coef': np.float64(0.0026349057392863044), 'stat': np.float64(4.981410192338361), 'pvalue': np.float64(6.312257761753051e-07), 'ref': 'neutral', 'test': 'positive'}, 'negative_vs_neutral': {'c

C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users

done
    all

    scores: {'baseline': np.float64(0.9981679034374863), 'positive': np.float64(0.9970971881455882), 'neutral': np.float64(0.996441213591801), 'negative': np.float64(0.9875409682655305)}
    sig:    {'baseline_vs_positive': {'coef': np.float64(-0.0010707152919018665), 'stat': np.float64(-1.463992581374764), 'pvalue': np.float64(0.14319597535286183), 'ref': 'baseline', 'test': 'positive'}, 'baseline_vs_neutral': {'coef': np.float64(-0.0017266898456928697), 'stat': np.float64(-2.0872867387078213), 'pvalue': np.float64(0.0368622228022775), 'ref': 'baseline', 'test': 'neutral'}, 'baseline_vs_negative': {'coef': np.float64(-0.010626935171963987), 'stat': np.float64(-6.465970173862346), 'pvalue': np.float64(1.0065093105710869e-10), 'ref': 'baseline', 'test': 'negative'}, 'positive_vs_neutral': {'coef': np.float64(0.0006559745537867024), 'stat': np.float64(1.9773074450147643), 'pvalue': np.float64(0.04800688864131082), 'ref': 'neutral', 'test': 'positive'}, 'negative_vs_neutral'

...

C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\andre\IdeaProjects\llm

done
    yta

    scores: {'baseline': np.float64(0.4876452843487899), 'positive': np.float64(0.23451688881837077), 'neutral': np.float64(0.22277090470132715), 'negative': np.float64(0.12510877089667755)}
    sig:    {'baseline_vs_positive': {'coef': np.float64(-0.2531283955304188), 'stat': np.float64(-17.28086380206273), 'pvalue': np.float64(6.555878911278147e-67), 'ref': 'baseline', 'test': 'positive'}, 'baseline_vs_neutral': {'coef': np.float64(-0.26487437964746463), 'stat': np.float64(-17.223509022633092), 'pvalue': np.float64(1.7692840399301728e-66), 'ref': 'baseline', 'test': 'neutral'}, 'baseline_vs_negative': {'coef': np.float64(-0.36253651345211607), 'stat': np.float64(-24.56201802684571), 'pvalue': np.float64(3.2181582250448943e-133), 'ref': 'baseline', 'test': 'negative'}, 'positive_vs_neutral': {'coef': np.float64(0.011745984117043634), 'stat': np.float64(2.0261995640075505), 'pvalue': np.float64(0.04274433831165064), 'ref': 'neutral', 'test': 'positive'}, 'negative_vs_neut

C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
C:\Users\andre\IdeaProjects\llm-moral-distractors-working

done
    yta

    scores: {'baseline': np.float64(0.061897838648608615), 'positive': np.float64(0.056839874959776514), 'neutral': np.float64(0.059365156293455526), 'negative': np.float64(0.05648244657914148)}
    sig:    {'baseline_vs_positive': {'coef': np.float64(-0.005057963688832157), 'stat': np.float64(-1.1267228449526254), 'pvalue': np.float64(0.25985968046040986), 'ref': 'baseline', 'test': 'positive'}, 'baseline_vs_neutral': {'coef': np.float64(-0.002532682355153125), 'stat': np.float64(-0.5727558312100675), 'pvalue': np.float64(0.56681002899833), 'ref': 'baseline', 'test': 'neutral'}, 'baseline_vs_negative': {'coef': np.float64(-0.005415392069467084), 'stat': np.float64(-0.9233263192483355), 'pvalue': np.float64(0.35583717610747134), 'ref': 'baseline', 'test': 'negative'}, 'positive_vs_neutral': {'coef': np.float64(-0.002525281333679027), 'stat': np.float64(-1.3246989195416117), 'pvalue': np.float64(0.1852710316803412), 'ref': 'neutral', 'test': 'positive'}, 'negative_vs_neutr

C:\Users\andre\IdeaProjects\llm-moral-distractors-working\venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## Result Tables

In [14]:
DATASET_META = {
    "moralchoice_high_ambiguity": {
        "score_label": "MMAP",
        "caption_detail": "high-ambiguity MoralChoice scenarios",
        "label": "tab:main-high-ambiguity",
        "options": ["all"]
    },
    "moralchoice_low_ambiguity": {
        "score_label": "MMAP",
        "caption_detail": "low-ambiguity MoralChoice scenarios",
        "label": "tab:main-low-ambiguity",
        "options": ["all"]
    },
    "normbank": {
        "score_label": "Score (good$-$bad)",
        "caption_detail": "NormBank scenarios",
        "label": "tab:main-normbank",
        "options": ["good", "ok", "bad"]
    },
    "reddit": {
        "score_label": "Score (NTA+NAH$-$YTA$-$ESH)",
        "caption_detail": "Reddit AITA scenarios",
        "label": "tab:main-reddit",
        "options": ["esh", "yta", "nta", "nah", "info"]
    },
}

tables = []
col_headers = ["Model", "Baseline", "Positive", "Neutral", "Negative"]

for dataset, model_results in all_results.items():
    meta = DATASET_META[dataset]
    for option in model_results:
        rows = []
        for model_key in model_results[option]:
            s = model_results[option][model_key]["mean_scores"]
            b   = s.get("baseline", float("nan"))
            pos = s.get("positive", float("nan"))
            neu = s.get("neutral",  float("nan"))
            neg = s.get("negative", float("nan"))

            colored = color_min_max({"baseline": b, "positive": pos, "neutral": neu, "negative": neg})
            rows.append([
                MODEL_LABEL.get(model_key, model_key),
                colored["baseline"],
                colored["positive"],
                colored["neutral"],
                colored["negative"],
            ])

        caption = f"{meta['caption_detail']} {option}."
        tables.append(build_latex_table(rows, col_headers, caption, meta["label"]))

output_path = OUTPUT_DIR / "main_results_tables.tex"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    f.write("\n\n".join(tables))

print(f"\nWrote {len(tables)} tables to {output_path}")


Wrote 10 tables to C:\Users\andre\IdeaProjects\llm-moral-distractors-working\fig\tables\main_results_tables.tex


## Significance Testing Tables

In [15]:
# Holm-Bonferroni corrections
pvals = []
for dataset, model_results in all_results.items():
    for option in model_results:
        rows = []
        for model_key in model_results[option]:
            sig_results = model_results[option][model_key]['sig']
            for comparison, stats in sig_results.items():
                pvals.append(stats['pvalue'])

corrected_sig_results = multipletests(pvals, alpha=0.05)
corrected_pvals = corrected_sig_results[1]
print(corrected_pvals)

[1.82704483e-002 5.68692491e-001 9.99427737e-001 6.32439467e-001
 7.48725548e-022 1.15219671e-034 3.54616251e-014 6.03914356e-014
 6.20282745e-023 1.00000000e+000 2.73557359e-005 3.93241978e-005
 1.43091998e-019 3.08187935e-006 8.68151708e-003 3.44135443e-019
 6.73675522e-106 1.92509207e-205 9.97216844e-001 9.99900531e-001
 9.99596737e-001 1.00000000e+000 6.92969765e-012 8.00828200e-014
 9.93550566e-001 9.99999999e-001 9.34748282e-001 6.59375228e-001
 1.67044315e-010 1.00845382e-019 1.41218382e-003 7.30345707e-004
 1.75259702e-001 1.00000000e+000 1.19410167e-001 3.75953684e-001
 2.35819116e-002 6.81647080e-001 6.80228317e-011 5.23900207e-001
 1.75515547e-117 6.78447312e-135 0.00000000e+000 0.00000000e+000
 0.00000000e+000 4.14200455e-019 0.00000000e+000 0.00000000e+000
 5.68415291e-014 5.53966446e-007 2.71855169e-105 9.02612407e-005
 0.00000000e+000 0.00000000e+000 9.99966319e-001 9.69589551e-001
 1.64061016e-008 9.86159840e-001 5.26452719e-051 6.25257890e-061
 2.92511750e-001 9.486923

In [16]:
# Table generation
tables = []
col_headers = [
    "Model",
    "Positive vs. Baseline", "Neutral vs. Baseline", "Negative vs. Baseline",
    "Positive vs. Neutral", "Negative vs. Neutral", "Positive vs. Negative"
]

i = 0
for dataset, model_results in all_results.items():
    meta = DATASET_META[dataset]
    for option in model_results:
        rows = []
        for model_key in model_results[option]:
            sig_results = model_results[option][model_key]['sig']
            row = [MODEL_LABEL.get(model_key, model_key)]
            for comparison, stats in sig_results.items():
                row.append(f"${fmt_sci(corrected_pvals[i])}$")
                i += 1
            rows.append(row)

        caption = f"{meta['caption_detail']} {option}."
        tables.append(build_latex_table(rows, col_headers, caption, meta["label"]))

output_path = OUTPUT_DIR / "p_values_tables.tex"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    f.write("\n\n".join(tables))

print(f"\nWrote {len(tables)} tables to {output_path}")


Wrote 10 tables to C:\Users\andre\IdeaProjects\llm-moral-distractors-working\fig\tables\p_values_tables.tex
